# 模块 02：边缘检测与特征匹配

边缘勾勒物体轮廓，特征点实现跨图像对应——这是全景拼接、AR、3D重建的基础。

本模块涵盖：经典边缘检测(Sobel/Canny/Laplacian)、现代特征检测(SIFT/ORB)、特征匹配与几何验证(FLANN/RANSAC)

> 预计学习时间：60-90 分钟

## 学习目标

- 理解图像梯度的概念和计算方法
- 掌握 Canny 边缘检测的三步骤
- 理解 SIFT 的尺度不变性原理
- 学会 FLANN 特征匹配
- 能用 RANSAC 剔除错误匹配

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage import data, filters, feature, color, transform
from scipy import ndimage
import cv2
from ipywidgets import interact, FloatSlider, IntSlider

img = data.camera()
img_color = data.astronaut()
# Create a rotated version for feature matching demo
img2 = np.rot90(data.camera(), k=1)

print("All imports OK!")
print(f"Image size: {img.shape}, Rotated: {img2.shape}")

## 1. 图像梯度与 Sobel 算子

Sobel 用两个 3x3 核分别计算水平和垂直梯度：
```
Gx (horizontal edges):     Gy (vertical edges):
[-1  0  +1]               [-1 -2 -1]
[-2  0  +2]               [ 0  0  0]
[-1  0  +1]               [+1 +2 +1]
```
梯度幅值 = sqrt(Gx^2 + Gy^2)

In [ ]:
edge_h = filters.sobel_h(img)  # Horizontal edges (vertical lines)
edge_v = filters.sobel_v(img)  # Vertical edges (horizontal lines)
edge_sobel = filters.sobel(img)  # Magnitude

fig, axes = plt.subplots(1, 4, figsize=(18,5))
axes[0].imshow(img, cmap='gray'); axes[0].set_title('Original',fontsize=13); axes[0].axis('off')
axes[1].imshow(edge_h, cmap='gray'); axes[1].set_title('Sobel H (Gx)\nDetects vertical edges',fontsize=12); axes[1].axis('off')
axes[2].imshow(edge_v, cmap='gray'); axes[2].set_title('Sobel V (Gy)\nDetects horizontal edges',fontsize=12); axes[2].axis('off')
axes[3].imshow(edge_sobel, cmap='gray'); axes[3].set_title('Sobel Magnitude',fontsize=13); axes[3].axis('off')
plt.tight_layout()
plt.show()

## 2. Canny 边缘检测——工业标准

Canny 四步骤：
1. 高斯平滑（去噪）
2. Sobel 梯度计算
3. 非极大值抑制（细化边缘）
4. 双阈值处理（强边缘/弱边缘/丢弃）

In [ ]:
def canny_demo(low=10, high=50, sigma=1.0):
    edges = feature.canny(img, sigma=sigma, low_threshold=low, high_threshold=high)
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(12,5))
    ax1.imshow(img,cmap='gray'); ax1.set_title('Original',fontsize=13); ax1.axis('off')
    ax2.imshow(edges,cmap='gray')
    ax2.set_title(f'Canny low={low} high={high} sigma={sigma}\n{edges.sum():,} edge pixels',fontsize=12); ax2.axis('off')
    plt.show()

interact(canny_demo,
    low=IntSlider(min=0,max=200,step=5,value=10),
    high=IntSlider(min=10,max=255,step=5,value=50),
    sigma=FloatSlider(min=0.1,max=3.0,step=0.1,value=1.0))
print("Adjust thresholds: higher high-threshold -> fewer but more confident edges")

## 3. 三种边缘检测方法对比

In [ ]:
img_g = color.rgb2gray(data.astronaut())

edge_sobel = filters.sobel(img_g)
edge_canny = feature.canny(img_g, sigma=1, low_threshold=30, high_threshold=80)
edge_laplace = np.abs(ndimage.laplace(img_g))
edge_laplace = edge_laplace / edge_laplace.max()

fig, axes = plt.subplots(1, 4, figsize=(18,5))
axes[0].imshow(img_g,cmap='gray'); axes[0].set_title('Grayscale Original',fontsize=12); axes[0].axis('off')
axes[1].imshow(edge_sobel,cmap='gray'); axes[1].set_title('Sobel (1st derivative)',fontsize=12); axes[1].axis('off')
axes[2].imshow(edge_canny,cmap='gray'); axes[2].set_title('Canny (NMS+dual threshold)',fontsize=12); axes[2].axis('off')
axes[3].imshow(edge_laplace,cmap='gray'); axes[3].set_title('Laplacian (2nd derivative)',fontsize=12); axes[3].axis('off')
plt.tight_layout()
plt.show()
for m,c in [('Sobel',edge_sobel.sum()),('Canny',edge_canny.sum()),('Laplacian',edge_laplace.sum())]:
    print(f"{m:10s}: {c:8,.0f} edge pixels")

## 4. SIFT 特征点——尺度不变特征变换

SIFT 是 CV 史上最重要的算法之一。每个特征点包含：
- **位置** (x,y)
- **尺度** (scale)——关键！不同尺度下的同一角点都能检测
- **方向** (orientation)——用于旋转不变性
- **描述子** (descriptor)——128维向量描述周围外观

In [ ]:
try:
    sift = cv2.SIFT_create()
    kp, des = sift.detectAndCompute(img, None)

    img_kp = cv2.drawKeypoints(img, kp, None,
        flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)

    plt.figure(figsize=(14,7))
    plt.imshow(img_kp)
    plt.title(f'SIFT Keypoints: {len(kp)} detected\n(Circle=scale, Line=orientation)',fontsize=13)
    plt.axis('off')
    plt.show()
    print(f"SIFT: {len(kp)} keypoints, descriptor shape: {des.shape} (128-dim each)")
except Exception as e:
    print(f"SIFT not available: {e}")
    print("Using ORB as fallback...")
    orb = cv2.ORB_create(nfeatures=500)
    kp, des = orb.detectAndCompute(img, None)
    img_kp = cv2.drawKeypoints(img, kp, None, color=(0,255,0))
    plt.figure(figsize=(14,7))
    plt.imshow(img_kp)
    plt.title(f'ORB Keypoints (fallback): {len(kp)} detected',fontsize=13)
    plt.axis('off')
    plt.show()

## 5. ORB——快速免费替代方案

| 特性 | SIFT | ORB |
|------|------|-----|
| 专利 | 有专利 | 免费 |
| 速度 | 较慢 | 极快(~100x) |
| 尺度不变 | Yes | No(图像金字塔弥补) |
| 旋转不变 | Yes | Yes |
| 描述子 | 128维浮点 | 32字节二进制 |

In [ ]:
orb = cv2.ORB_create(nfeatures=500)
kp1, des1 = orb.detectAndCompute(img, None)
kp2, des2 = orb.detectAndCompute(img2, None)

fig, axes = plt.subplots(1, 2, figsize=(14,6))
for ax, kp_i, title, color in [
    (axes[0], kp1, f'Image 1: {len(kp1)} keypoints', (0,255,0)),
    (axes[1], kp2, f'Image 2 (rotated): {len(kp2)} keypoints', (0,255,0))
]:
    drawn = cv2.drawKeypoints(img if '1' in title else img2, kp_i, None, color=color)
    ax.imshow(drawn); ax.set_title(title,fontsize=13); ax.axis('off')
plt.tight_layout()
plt.show()

## 6. 特征匹配 + Ratio Test

找到两张图片之间特征点的对应关系。Lowe's ratio test: 最佳匹配距离必须显著小于次佳匹配（ratio<0.75），否则视为不可靠。

In [ ]:
bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False)
matches = bf.knnMatch(des1, des2, k=2)

# Lowe's ratio test
good = [m for m,n in matches if m.distance < 0.75*n.distance]

fig, axes = plt.subplots(1, 2, figsize=(16,7))
axes[0].imshow(cv2.drawMatches(img,kp1,img2,kp2,matches[:50],None))
axes[0].set_title(f'All matches: {len(matches)}',fontsize=12); axes[0].axis('off')
axes[1].imshow(cv2.drawMatches(img,kp1,img2,kp2,good[:50],None))
axes[1].set_title(f'Ratio test filtered: {len(good)} (kept {100*len(good)/len(matches):.0f}%)',fontsize=12); axes[1].axis('off')
plt.tight_layout()
plt.show()
print(f"Ratio test kept {100*len(good)/len(matches):.0f}% of matches")

## 7. RANSAC——几何一致性验证

RANSAC (Random Sample Consensus) 通过几何模型进一步剔除误匹配：
1. 随机采样最小点集 → 拟合模型(如单应矩阵)
2. 统计支持该模型的"内点"
3. 反复迭代，保留内点最多的模型

In [ ]:
if len(good) >= 10:
    src_pts = np.float32([kp1[m.queryIdx].pt for m in good]).reshape(-1,1,2)
    dst_pts = np.float32([kp2[m.trainIdx].pt for m in good]).reshape(-1,1,2)

    H, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, ransacReprojThreshold=5.0)
    inliers = mask.ravel().tolist()
    inlier_matches = [m for m,ok in zip(good, inliers) if ok]

    img_ransac = cv2.drawMatches(img,kp1,img2,kp2,inlier_matches[:50],None)
    plt.figure(figsize=(14,7))
    plt.imshow(img_ransac)
    plt.title(f'RANSAC: {sum(inliers)} inliers / {len(good)} total ({100*sum(inliers)/len(good):.0f}%)',fontsize=13)
    plt.axis('off'); plt.show()

    print(f"RANSAC results: {sum(inliers)} inliers kept, {len(good)-sum(inliers)} outliers removed")
    print(f"Homography matrix:\n{H}")
else:
    print(f"Not enough good matches ({len(good)}), need >= 10")

## 📝 应用案例：全景拼接流程

全景拼接 = 特征检测 → 匹配 → 单应矩阵 → 变形 → 融合。完整交互Demo请运行：
```bash
python demos/panorama_stitcher.py
```

In [ ]:
# Quick homography demo: perspective warp
src_pts = np.float32([[0,0],[img.shape[1]-1,0],[img.shape[1]-1,img.shape[0]-1],[0,img.shape[0]-1]])
dst_pts = np.float32([[20,20],[img.shape[1]-40,10],[img.shape[1]-30,img.shape[0]-30],[10,img.shape[0]-20]])

H_simple,_ = cv2.findHomography(src_pts, dst_pts)
warped = cv2.warpPerspective(img, H_simple, (img.shape[1],img.shape[0]))

fig, axes = plt.subplots(1,2,figsize=(14,6))
axes[0].imshow(img,cmap='gray'); axes[0].set_title('Original',fontsize=13); axes[0].axis('off')
axes[1].imshow(warped,cmap='gray'); axes[1].set_title('After Homography Warp',fontsize=13); axes[1].axis('off')
plt.tight_layout(); plt.show()
print("Run demos/panorama_stitcher.py for full interactive panorama stitching!")

## 🏋️ 动手练习

1. **Canny阈值实验**：对astronaut图尝试极端阈值组合(high=200,low=5 vs high=80,low=50)
2. **ORB vs SIFT**：统计同一张图上两种方法的特征点数量和分布差异
3. **Ratio test阈值**：调整ratio阈值(0.5~0.9)观察对匹配数量的影响

In [ ]:
# TODO: Exercise 1 - Canny extreme threshold comparison
# Run Canny with Group A (low=5, high=200) vs Group B (low=80, high=100)
# Compare edge quality

# TODO: Exercise 3 - Ratio test threshold experiment
# Try ratios [0.5, 0.6, 0.7, 0.8, 0.9] and plot match count vs ratio

print("Complete the exercises above!")

## 📖 本节总结

| 概念 | 核心要点 |
|------|---------|
| 梯度 | 像素变化率，边缘=梯度大的位置 |
| Sobel | 一阶导数，H+V两个方向 |
| Canny | 平滑->梯度->NMS->双阈值，工业标准 |
| SIFT | 尺度不变,128维浮点描述子 |
| ORB | 快速免费替代,32字节二进制 |
| FLANN | 快速近似最近邻匹配 |
| RANSAC | 几何一致性验证剔除误匹配 |

下一课：模块03 — 几何变换实战